# MVTec AD — Dataset Exploration

This notebook inventories all 15 MVTec AD categories, checks the normal-only train/validation split, inspects official test defect types and masks, and runs loader sanity checks. The official test set is not used for training or validation.

In [ ]:
# 1. Mount Drive and clone the project repository
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/pranitas-cloud/CS-663.git
%cd /content/CS-663

In [ ]:
# 2. Imports and dataset location
from pathlib import Path
from collections import Counter
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, '/content/CS-663')
from src.data.dataset import MVTecDataset, available_categories

DATA_ROOT = Path('/content/drive/MyDrive/CS663_Shared/datasets/mvtec_anomaly_detection')
assert DATA_ROOT.is_dir(), f'Dataset folder not found: {DATA_ROOT}'
categories = available_categories(DATA_ROOT)
print(f'Found {len(categories)} categories:')
print(categories)
assert len(categories) == 15, f'Expected 15 categories, found {len(categories)}'
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Inventory all categories
Count normal training images, test images by defect type, and available ground-truth masks.

In [ ]:
rows = []
for category in categories:
    base = DATA_ROOT / category
    n_train = len(list((base / 'train' / 'good').glob('*.png')))
    defect_counts = {
        folder.name: len(list(folder.glob('*.png')))
        for folder in sorted((base / 'test').iterdir()) if folder.is_dir()
    }
    n_test = sum(defect_counts.values())
    n_masks = sum(len(list((base / 'ground_truth' / defect).glob('*_mask.png')))
                  for defect in defect_counts if defect != 'good'
                  and (base / 'ground_truth' / defect).is_dir())
    rows.append({'category': category, 'train_good': n_train,
                 'test_total': n_test, 'test_good': defect_counts.get('good', 0),
                 'test_anomalous': n_test - defect_counts.get('good', 0),
                 'ground_truth_masks': n_masks,
                 'test_defect_types': ', '.join(k for k in defect_counts if k != 'good')})

inventory = pd.DataFrame(rows)
display(inventory)
print('\nTotal normal training images:', inventory['train_good'].sum())
print('Total test images:', inventory['test_total'].sum())
print('Total anomalous test images:', inventory['test_anomalous'].sum())

In [ ]:
# 4. Visualize the number of images per category
ax = inventory.set_index('category')[['train_good', 'test_total']].plot(
    kind='bar', figsize=(14, 5), rot=45)
ax.set_ylabel('Number of images')
ax.set_title('MVTec AD image counts by category')
plt.tight_layout()
plt.show()

## 5. Verify the train/validation/test split
We hold out 10% of `train/good` for validation using a fixed seed. The official test set remains unchanged. Start with `bottle` as a pilot category.

In [ ]:
CATEGORY = 'bottle'
train_ds = MVTecDataset(DATA_ROOT, CATEGORY, split='train', val_fraction=0.10, seed=42)
val_ds = MVTecDataset(DATA_ROOT, CATEGORY, split='val', val_fraction=0.10, seed=42)
test_ds = MVTecDataset(DATA_ROOT, CATEGORY, split='test')

train_paths = {s[0] for s in train_ds.samples}
val_paths = {s[0] for s in val_ds.samples}
test_paths = {s[0] for s in test_ds.samples}
print(f'{CATEGORY}: train-good={len(train_ds)}, val-good={len(val_ds)}, test={len(test_ds)}')
print('Train/validation overlap:', len(train_paths & val_paths))
print('Train/test overlap:', len(train_paths & test_paths))
print('Validation/test overlap:', len(val_paths & test_paths))
print('Test images by defect type:', dict(Counter(s[2] for s in test_ds.samples)))
assert not (train_paths & val_paths)
assert not (train_paths & test_paths)
assert not (val_paths & test_paths)
print('Split checks passed.')

## 6. Visualize representative test images and masks
For anomalous test images, the mask marks the annotated defect pixels. Normal (`good`) images have all-zero masks.

In [ ]:
good_ids = [i for i, s in enumerate(test_ds.samples) if s[2] == 'good']
bad_ids = [i for i, s in enumerate(test_ds.samples) if s[2] != 'good']
chosen = good_ids[:2] + bad_ids[:4]
chosen = chosen[:6]

fig, axes = plt.subplots(len(chosen), 2, figsize=(9, 3.2 * len(chosen)))
if len(chosen) == 1:
    axes = np.expand_dims(axes, axis=0)
for row, idx in enumerate(chosen):
    sample = test_ds[idx]
    image = sample['image'].permute(1, 2, 0).numpy()
    mask = sample['mask'].squeeze(0).numpy()
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"Test image — {sample['defect_type']}")
    axes[row, 1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f"Ground-truth mask — {int(mask.sum())} defect pixels")
    for ax in axes[row]: ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Loader sanity checks (shape, range, mask values, and mask availability)
sample = test_ds[0]
assert sample['image'].shape == (3, 256, 256), sample['image'].shape
assert sample['mask'].shape == (1, 256, 256), sample['mask'].shape
assert sample['image'].dtype == torch.float32
assert 0.0 <= sample['image'].min().item() <= sample['image'].max().item() <= 1.0
assert set(torch.unique(sample['mask']).tolist()).issubset({0.0, 1.0})

# Check every anomalous test sample has a corresponding mask file.
missing_masks = [str(mask) for _, mask, defect in test_ds.samples
                 if defect != 'good' and (mask is None or not mask.is_file())]
print('Missing anomalous masks:', len(missing_masks))
assert not missing_masks, missing_masks[:5]
print('All loader sanity checks passed.')

## 8. What we record from exploration
- Category-wise normal training and test image counts.
- Defect types and ground-truth mask availability.
- Deterministic 90/10 normal-only train/validation split.
- Image/mask dimensions, numeric ranges, and representative examples.

Once these checks pass, we can move on to PCA experiments. No model has been trained in this notebook.